# Model Pipeline Benchmark

Clean notebook for:
1. Optional training
2. Validation
3. ONNX export (FP32 + FP16)
4. Latency/FPS benchmark
5. Final comparison table


In [1]:
from pathlib import Path
import time
import shutil
from typing import Any

import cv2
import numpy as np
import pandas as pd
import torch
import yaml
import onnxruntime as ort
from ultralytics import YOLO


In [2]:
# ==============================
# Global settings
# ==============================

# Pipeline switches
RUN_TRAIN = False
RUN_VALIDATE = True
RUN_EXPORT = True
RUN_BENCHMARK = True
SAVE_REPORTS = True

# Benchmark settings
BENCHMARK_WARMUP = 10
BENCHMARK_RUNS = 80
MAX_BENCHMARK_IMAGES = None  # None -> use full split


def detect_project_root() -> Path:
    # Find project root by configs/train_config.yaml
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "configs" / "train_config.yaml").exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not locate project root from cwd: {Path.cwd()}")


PROJECT_ROOT = detect_project_root()
TRAIN_CONFIG_PATH = PROJECT_ROOT / "configs" / "train_config.yaml"

with TRAIN_CONFIG_PATH.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f) or {}

train_cfg = cfg.get("train", {})
eval_cfg = cfg.get("evaluate", {})
export_cfg = cfg.get("export", {})

# Devices are taken from config
TRAIN_DEVICE = str(train_cfg.get("device", "cpu"))
EVAL_DEVICE = str(eval_cfg.get("device", TRAIN_DEVICE))
EXPORT_DEVICE = str(export_cfg.get("device", TRAIN_DEVICE))
BENCHMARK_DEVICE = EVAL_DEVICE

# Core params from config
EVAL_SPLIT = str(eval_cfg.get("split", "test"))
IMGSZ = int(eval_cfg.get("imgsz", train_cfg.get("imgsz", 640)))
BATCH = int(eval_cfg.get("batch", train_cfg.get("batch", 16)))

# Paths from config
weights_path = (PROJECT_ROOT / eval_cfg.get("weights", "models/weights/best.pt")).resolve()
dataset_yaml = (PROJECT_ROOT / eval_cfg.get("data", train_cfg.get("data", "configs/dataset.yaml"))).resolve()

export_dir = (PROJECT_ROOT / export_cfg.get("output_dir", "models/onnx")).resolve()
export_dir.mkdir(parents=True, exist_ok=True)
onnx_path = export_dir / "model.onnx"
onnx_fp16_path = export_dir / "model.fp16.onnx"

reports_dir = (PROJECT_ROOT / "reports").resolve()
reports_dir.mkdir(parents=True, exist_ok=True)
dataset_yaml_runtime = reports_dir / "dataset_abs.yaml"

print("project_root:", PROJECT_ROOT)
print("train_config:", TRAIN_CONFIG_PATH)
print("weights:", weights_path)
print("dataset_yaml:", dataset_yaml)
print("devices -> train/eval/export/bench:", TRAIN_DEVICE, EVAL_DEVICE, EXPORT_DEVICE, BENCHMARK_DEVICE)
print("run flags -> train/validate/export/benchmark:", RUN_TRAIN, RUN_VALIDATE, RUN_EXPORT, RUN_BENCHMARK)


project_root: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline
train_config: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\configs\train_config.yaml
weights: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\models\weights\best.pt
dataset_yaml: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\data\processed\YOLO\dataset.yaml
devices -> train/eval/export/bench: cuda cuda cuda cuda
run flags -> train/validate/export/benchmark: False True True True


In [3]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def parse_val_metrics(results) -> dict[str, Any]:
    # Extract key metrics from Ultralytics validation output
    metrics = {}
    box = getattr(results, "box", None)
    if box is not None:
        for attr, name in [("mp", "precision"), ("mr", "recall"), ("map50", "map50"), ("map", "map50_95")]:
            value = getattr(box, attr, None)
            if value is not None:
                metrics[name] = float(value)
    return metrics


def model_size_mb(path: Path) -> float:
    return path.stat().st_size / (1024 * 1024) if path.exists() else float("nan")


def rel_path(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT)).replace("\\", "/")
    except Exception:
        return path.resolve().as_posix()


def _resolve_data_root(dataset_cfg: dict, dataset_yaml_path: Path) -> Path:
    root = Path(dataset_cfg.get("path", dataset_yaml_path.parent))
    if not root.is_absolute():
        root = (PROJECT_ROOT / root).resolve()
    return root


def write_runtime_dataset_yaml(source_yaml: Path, target_yaml: Path) -> Path:
    # Create dataset yaml with absolute path to avoid cwd/platform path issues
    with source_yaml.open("r", encoding="utf-8") as f:
        data_cfg = yaml.safe_load(f) or {}

    root = _resolve_data_root(data_cfg, source_yaml)
    data_cfg["path"] = root.as_posix()

    target_yaml.parent.mkdir(parents=True, exist_ok=True)
    with target_yaml.open("w", encoding="utf-8") as f:
        yaml.safe_dump(data_cfg, f, allow_unicode=True, sort_keys=False)

    return target_yaml


def get_split_images(dataset_yaml_path: Path, split_name: str, max_images: int | None = None) -> list[Path]:
    with dataset_yaml_path.open("r", encoding="utf-8") as f:
        data_cfg = yaml.safe_load(f) or {}

    root = _resolve_data_root(data_cfg, dataset_yaml_path)
    split_rel = data_cfg.get(split_name)
    if split_rel is None:
        raise ValueError(f"Split '{split_name}' not found in {dataset_yaml_path}")

    split_dir = (root / split_rel).resolve()
    if not split_dir.exists():
        raise FileNotFoundError(f"Split directory not found: {split_dir}")

    images = sorted([p for p in split_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
    return images if max_images is None else images[:max_images]


def read_unicode_image(path: Path):
    data = np.fromfile(str(path), dtype=np.uint8)
    if data.size == 0:
        return None
    return cv2.imdecode(data, cv2.IMREAD_COLOR)


def preprocess_for_onnx(image_path: Path, size: int) -> np.ndarray:
    image = read_unicode_image(image_path)
    if image is None:
        raise ValueError(f"Cannot read image: {image_path}")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (size, size), interpolation=cv2.INTER_LINEAR)
    image = image.astype(np.float32) / 255.0
    image = np.transpose(image, (2, 0, 1))[None, ...]
    return image


def _torch_device_name(device_name: str) -> str:
    d = str(device_name).lower().strip()
    if d.isdigit():
        return f"cuda:{d}"
    if d.startswith("cuda"):
        return d
    return "cpu"


def _onnx_providers_for_device(device_name: str) -> list[str]:
    available = ort.get_available_providers()
    d = str(device_name).lower().strip()
    want_cuda = d.isdigit() or d.startswith("cuda")

    if want_cuda:
        if "CUDAExecutionProvider" not in available:
            raise RuntimeError(
                f"CUDAExecutionProvider is unavailable in ONNX Runtime. Available providers: {available}"
            )
        return ["CUDAExecutionProvider", "CPUExecutionProvider"]

    return ["CPUExecutionProvider"]


def _onnx_expected_dtype(input_type: str):
    t = (input_type or "").lower()
    if "float16" in t:
        return np.float16
    if "double" in t:
        return np.float64
    return np.float32


def benchmark_pt(weights: Path, tensors: list[np.ndarray], device_name: str, warmup: int, runs: int) -> dict[str, Any]:
    dev_name = _torch_device_name(device_name)
    dev = torch.device(dev_name if (dev_name.startswith("cuda") and torch.cuda.is_available()) else "cpu")

    model = YOLO(str(weights), task="detect").model.to(dev).eval()
    torch_tensors = [torch.from_numpy(t).to(dev) for t in tensors]

    with torch.no_grad():
        for i in range(warmup):
            _ = model(torch_tensors[i % len(torch_tensors)])
        if dev.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        for i in range(runs):
            _ = model(torch_tensors[i % len(torch_tensors)])
        if dev.type == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()

    latency_ms = (t1 - t0) * 1000.0 / runs
    return {
        "runtime": f"torch:{dev.type}",
        "latency_ms": latency_ms,
        "fps": (1000.0 / latency_ms) if latency_ms > 0 else float("nan"),
        "input_dtype": "float32",
    }


def benchmark_onnx(model_path: Path, tensors: list[np.ndarray], device_name: str, warmup: int, runs: int) -> dict[str, Any]:
    providers = _onnx_providers_for_device(device_name)
    sess = ort.InferenceSession(str(model_path), providers=providers)

    input_meta = sess.get_inputs()[0]
    input_name = input_meta.name
    expected_dtype = _onnx_expected_dtype(getattr(input_meta, "type", "tensor(float)"))
    cast_tensors = [np.asarray(t, dtype=expected_dtype) for t in tensors]

    for i in range(warmup):
        _ = sess.run(None, {input_name: cast_tensors[i % len(cast_tensors)]})

    t0 = time.perf_counter()
    for i in range(runs):
        _ = sess.run(None, {input_name: cast_tensors[i % len(cast_tensors)]})
    t1 = time.perf_counter()

    latency_ms = (t1 - t0) * 1000.0 / runs
    return {
        "runtime": "onnxruntime:" + ",".join(sess.get_providers()),
        "latency_ms": latency_ms,
        "fps": (1000.0 / latency_ms) if latency_ms > 0 else float("nan"),
        "input_dtype": str(expected_dtype).replace("<class '", "").replace("'>", ""),
    }


In [4]:
# Step 1: prepare runtime dataset config and benchmark tensors

dataset_yaml_runtime = write_runtime_dataset_yaml(dataset_yaml, dataset_yaml_runtime)
sample_images = get_split_images(dataset_yaml_runtime, split_name=EVAL_SPLIT, max_images=MAX_BENCHMARK_IMAGES)
if not sample_images:
    raise RuntimeError(f"No images found in split '{EVAL_SPLIT}'")

input_tensors = [preprocess_for_onnx(p, IMGSZ) for p in sample_images]

print("dataset_yaml_runtime:", dataset_yaml_runtime)
print("split:", EVAL_SPLIT)
print("images used for benchmark:", len(sample_images))
print("imgsz:", IMGSZ)


dataset_yaml_runtime: C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\reports\dataset_abs.yaml
split: test
images used for benchmark: 579
imgsz: 640


In [5]:
# Step 2: optional training

if RUN_TRAIN:
    train_args = dict(train_cfg)
    train_args["data"] = str(dataset_yaml_runtime)

    # Keep training runs inside project directory
    if "project" in train_args:
        train_args["project"] = str((PROJECT_ROOT / str(train_args["project"])).resolve())

    model_id = train_args.pop("model", "yolo11n.pt")
    trainer = YOLO(str(model_id), task="detect")
    _ = trainer.train(**train_args)

    print("Training finished.")
else:
    print("RUN_TRAIN=False -> training skipped")


RUN_TRAIN=False -> training skipped


In [6]:
# Step 3: validation (.pt)

if not weights_path.exists():
    raise FileNotFoundError(f"Weights not found: {weights_path}")

pt_metrics = {}
pt_model = YOLO(str(weights_path), task="detect")

if RUN_VALIDATE:
    pt_val = pt_model.val(
        data=str(dataset_yaml_runtime),
        split=EVAL_SPLIT,
        imgsz=IMGSZ,
        batch=BATCH,
        device=EVAL_DEVICE,
        verbose=False,
    )
    pt_metrics = parse_val_metrics(pt_val)

print("pt_metrics:", pt_metrics)


Ultralytics 8.3.145  Python-3.12.3 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce GTX 1080, 8192MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 693.7160.3 MB/s, size: 67.6 KB)


val: Scanning C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\data\processed\YOLO\labels\test.cache... 579 images, 92 backgrounds, 0 corrupt: 100%|██████████| 579/579 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 37/37 [00:04<00:00,  9.01it/s]


                   all        579        845      0.513      0.347      0.356      0.166
Speed: 0.2ms preprocess, 2.7ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to runs\detect\val
pt_metrics: {'precision': 0.512575668682112, 'recall': 0.3467455621301775, 'map50': 0.3557961578292047, 'map50_95': 0.16581011196793313}


In [7]:
# Step 4: export ONNX (FP32 + FP16)

if RUN_EXPORT:
    exported = pt_model.export(
        format="onnx",
        imgsz=int(export_cfg.get("imgsz", IMGSZ)),
        dynamic=bool(export_cfg.get("dynamic", False)),
        simplify=bool(export_cfg.get("simplify", True)),
        half=False,
        device=EXPORT_DEVICE,
    )
    exported_path = Path(str(exported)).resolve()
    if exported_path != onnx_path:
        shutil.copy2(exported_path, onnx_path)

    try:
        exported_fp16 = pt_model.export(
            format="onnx",
            imgsz=int(export_cfg.get("imgsz", IMGSZ)),
            dynamic=bool(export_cfg.get("dynamic", False)),
            simplify=bool(export_cfg.get("simplify", True)),
            half=True,
            device=EXPORT_DEVICE,
        )
        exported_fp16_path = Path(str(exported_fp16)).resolve()
        if exported_fp16_path != onnx_fp16_path:
            shutil.copy2(exported_fp16_path, onnx_fp16_path)
    except Exception as exc:
        print("FP16 export failed:", repr(exc))

print("onnx_path:", onnx_path, "exists:", onnx_path.exists())
print("onnx_fp16_path:", onnx_fp16_path, "exists:", onnx_fp16_path.exists())


Ultralytics 8.3.145  Python-3.12.3 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce GTX 1080, 8192MiB)

PyTorch: starting from 'C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.2 MB)

ONNX: starting export with onnx 1.17.0 opset 19...
ONNX: slimming with onnxslim 0.1.58...
ONNX: export success  1.2s, saved as 'C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\weights\best.onnx' (10.1 MB)

Export complete (1.3s)
Results saved to C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\weights
Predict:         yolo predict task=detect model=C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\weights\best.onnx imgsz=640  
Validate:        yolo val task=detect model=C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\weights\best.onnx imgsz=640 data=data/train_data/data.yaml  
Visualize:       https://netron.app
Ultralytics 8.3.145  Python-3.12.3 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce GTX 1080, 8192MiB)

PyTor

In [8]:
# Step 5: validation (ONNX formats)

onnx_metrics = {}
onnx_fp16_metrics = {}

if RUN_VALIDATE and onnx_path.exists():
    try:
        onnx_model = YOLO(str(onnx_path), task="detect")
        onnx_val = onnx_model.val(
            data=str(dataset_yaml_runtime),
            split=EVAL_SPLIT,
            imgsz=IMGSZ,
            batch=BATCH,
            device=EVAL_DEVICE,
            verbose=False,
        )
        onnx_metrics = parse_val_metrics(onnx_val)
    except Exception as exc:
        print("ONNX validation failed:", repr(exc))

if RUN_VALIDATE and onnx_fp16_path.exists():
    try:
        onnx_fp16_model = YOLO(str(onnx_fp16_path), task="detect")
        onnx_fp16_val = onnx_fp16_model.val(
            data=str(dataset_yaml_runtime),
            split=EVAL_SPLIT,
            imgsz=IMGSZ,
            batch=BATCH,
            device=EVAL_DEVICE,
            verbose=False,
        )
        onnx_fp16_metrics = parse_val_metrics(onnx_fp16_val)
    except Exception as exc:
        print("ONNX FP16 validation failed:", repr(exc))

print("onnx_metrics:", onnx_metrics)
print("onnx_fp16_metrics:", onnx_fp16_metrics)


Ultralytics 8.3.145  Python-3.12.3 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce GTX 1080, 8192MiB)
Loading C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\onnx\model.onnx for ONNX Runtime inference...
Using ONNX Runtime CUDAExecutionProvider
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access  (ping: 0.00.0 ms, read: 763.1470.2 MB/s, size: 73.1 KB)


val: Scanning C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\data\processed\YOLO\labels\test.cache... 579 images, 92 backgrounds, 0 corrupt: 100%|██████████| 579/579 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 579/579 [00:07<00:00, 81.80it/s]


                   all        579        845      0.494      0.349      0.354      0.165
Speed: 0.4ms preprocess, 7.7ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to runs\detect\val2
Ultralytics 8.3.145  Python-3.12.3 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce GTX 1080, 8192MiB)
Loading C:\Users\sanek\OneDrive\ \cv_drone_pipeline\models\onnx\model.fp16.onnx for ONNX Runtime inference...
Using ONNX Runtime CUDAExecutionProvider
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access  (ping: 0.00.0 ms, read: 1389.4412.8 MB/s, size: 141.0 KB)


val: Scanning C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\data\processed\YOLO\labels\test.cache... 579 images, 92 backgrounds, 0 corrupt: 100%|██████████| 579/579 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 579/579 [00:10<00:00, 56.60it/s]


                   all        579        845      0.494      0.353      0.356      0.166
Speed: 0.4ms preprocess, 12.0ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to runs\detect\val3
onnx_metrics: {'precision': 0.4940473274093936, 'recall': 0.34911242603550297, 'map50': 0.35404841608608706, 'map50_95': 0.1648364954979808}
onnx_fp16_metrics: {'precision': 0.4943899975008231, 'recall': 0.3529373160142391, 'map50': 0.3562938226983525, 'map50_95': 0.16561640089194518}


In [9]:
# Step 6: benchmark + final comparison table

comparison_df = pd.DataFrame()

if RUN_BENCHMARK:
    rows: list[dict[str, Any]] = []

    pt_bench = benchmark_pt(
        weights=weights_path,
        tensors=input_tensors,
        device_name=BENCHMARK_DEVICE,
        warmup=BENCHMARK_WARMUP,
        runs=BENCHMARK_RUNS,
    )
    rows.append(
        {
            "model_format": ".pt",
            "path": rel_path(weights_path),
            "size_mb": round(model_size_mb(weights_path), 2),
            "precision": round(float(pt_metrics.get("precision", float("nan"))), 4),
            "recall": round(float(pt_metrics.get("recall", float("nan"))), 4),
            "map50": round(float(pt_metrics.get("map50", float("nan"))), 4),
            "map50_95": round(float(pt_metrics.get("map50_95", float("nan"))), 4),
            "latency_ms": round(float(pt_bench["latency_ms"]), 3),
            "fps": round(float(pt_bench["fps"]), 2),
            "runtime": pt_bench["runtime"],
            "input_dtype": pt_bench["input_dtype"],
        }
    )

    if onnx_path.exists():
        onnx_bench = benchmark_onnx(
            model_path=onnx_path,
            tensors=input_tensors,
            device_name=BENCHMARK_DEVICE,
            warmup=BENCHMARK_WARMUP,
            runs=BENCHMARK_RUNS,
        )
        rows.append(
            {
                "model_format": ".onnx",
                "path": rel_path(onnx_path),
                "size_mb": round(model_size_mb(onnx_path), 2),
                "precision": round(float(onnx_metrics.get("precision", float("nan"))), 4),
                "recall": round(float(onnx_metrics.get("recall", float("nan"))), 4),
                "map50": round(float(onnx_metrics.get("map50", float("nan"))), 4),
                "map50_95": round(float(onnx_metrics.get("map50_95", float("nan"))), 4),
                "latency_ms": round(float(onnx_bench["latency_ms"]), 3),
                "fps": round(float(onnx_bench["fps"]), 2),
                "runtime": onnx_bench["runtime"],
                "input_dtype": onnx_bench["input_dtype"],
            }
        )

    if onnx_fp16_path.exists():
        onnx_fp16_bench = benchmark_onnx(
            model_path=onnx_fp16_path,
            tensors=input_tensors,
            device_name=BENCHMARK_DEVICE,
            warmup=BENCHMARK_WARMUP,
            runs=BENCHMARK_RUNS,
        )
        rows.append(
            {
                "model_format": ".onnx (fp16)",
                "path": rel_path(onnx_fp16_path),
                "size_mb": round(model_size_mb(onnx_fp16_path), 2),
                "precision": round(float(onnx_fp16_metrics.get("precision", float("nan"))), 4),
                "recall": round(float(onnx_fp16_metrics.get("recall", float("nan"))), 4),
                "map50": round(float(onnx_fp16_metrics.get("map50", float("nan"))), 4),
                "map50_95": round(float(onnx_fp16_metrics.get("map50_95", float("nan"))), 4),
                "latency_ms": round(float(onnx_fp16_bench["latency_ms"]), 3),
                "fps": round(float(onnx_fp16_bench["fps"]), 2),
                "runtime": onnx_fp16_bench["runtime"],
                "input_dtype": onnx_fp16_bench["input_dtype"],
            }
        )

    comparison_df = pd.DataFrame(rows).sort_values("latency_ms")

comparison_df


,model_format,path,size_mb,precision,recall,map50,map50_95,latency_ms,fps,runtime,input_dtype
1,.onnx,models/onnx/model.onnx,10.09,0.4940,0.3491,0.3540,0.1648,9.332,107.16,"onnxruntime:CUDAExecutionProvider,CPUExecution...",numpy.float32
0,.pt,models/weights/best.pt,5.19,0.5126,0.3467,0.3558,0.1658,12.505,79.97,torch:cuda,float32
2,.onnx (fp16),models/onnx/model.fp16.onnx,5.09,0.4944,0.3529,0.3563,0.1656,12.812,78.05,"onnxruntime:CUDAExecutionProvider,CPUExecution...",numpy.float16


In [10]:
# Step 7: save benchmark report

if SAVE_REPORTS and not comparison_df.empty:
    d = str(BENCHMARK_DEVICE).lower().strip()
    device_tag = "gpu" if (d.startswith("cuda") or d.isdigit()) else "cpu"

    out_csv = reports_dir / f"model_comparison_{device_tag}.csv"
    out_md = reports_dir / f"model_comparison_{device_tag}.md"

    comparison_df.to_csv(out_csv, index=False)
    with out_md.open("w", encoding="utf-8") as f:
        f.write(comparison_df.to_markdown(index=False))

    print("Saved:")
    print("-", out_csv)
    print("-", out_md)
else:
    print("Skip saving reports (empty table or SAVE_REPORTS=False)")


Saved:
- C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\reports\model_comparison_gpu.csv
- C:\Users\sanek\OneDrive\Рабочий стол\cv_drone_pipeline\reports\model_comparison_gpu.md


## Notes

- Edit only the **Global settings** block to control notebook behavior.
- Devices are read from `configs/train_config.yaml`.
- Use `MAX_BENCHMARK_IMAGES` if you need faster iterations while debugging.


In [ ]:
# Step 8: final conclusion and production recommendation

if comparison_df.empty:
    raise RuntimeError("comparison_df is empty. Run benchmark step first.")

pt_row = comparison_df.loc[comparison_df["model_format"] == ".pt"].iloc[0]
onnx_row = comparison_df.loc[comparison_df["model_format"] == ".onnx"].iloc[0] if (comparison_df["model_format"] == ".onnx").any() else None
onnx_fp16_row = comparison_df.loc[comparison_df["model_format"] == ".onnx (fp16)"].iloc[0] if (comparison_df["model_format"] == ".onnx (fp16)").any() else None

summary_rows = []
for _, row in comparison_df.iterrows():
    summary_rows.append({
        "model_format": row["model_format"],
        "fps": round(float(row["fps"]), 2),
        "latency_ms": round(float(row["latency_ms"]), 3),
        "map50": round(float(row["map50"]), 4),
        "map50_95": round(float(row["map50_95"]), 4),
        "fps_vs_pt_x": round(float(row["fps"]) / float(pt_row["fps"]), 3) if float(pt_row["fps"]) > 0 else float("nan"),
        "delta_map50_vs_pt": round(float(row["map50"]) - float(pt_row["map50"]), 4),
        "delta_map50_95_vs_pt": round(float(row["map50_95"]) - float(pt_row["map50_95"]), 4),
    })

decision_df = pd.DataFrame(summary_rows).sort_values("fps", ascending=False)
decision_df

recommended_model = ".onnx" if onnx_row is not None else ".pt"
print("Recommended production edge model:", recommended_model)

if onnx_row is not None:
    print(f"ONNX vs PT speedup: {float(onnx_row['fps'])/float(pt_row['fps']):.2f}x")
    print(f"ONNX vs PT delta mAP50: {float(onnx_row['map50']) - float(pt_row['map50']):+.4f}")
    print(f"ONNX vs PT delta mAP50-95: {float(onnx_row['map50_95']) - float(pt_row['map50_95']):+.4f}")

if onnx_fp16_row is not None and onnx_row is not None:
    print(f"ONNX FP16 vs ONNX FP32 FPS ratio: {float(onnx_fp16_row['fps'])/float(onnx_row['fps']):.2f}x")



## Final Conclusions

**Production edge-inference choice: `.onnx` (FP32).**

Why:

- In the current benchmark, `.onnx` has the best speed profile (lowest latency and highest FPS).
- Quality drop vs `.pt` is small:
  - `mAP50`: `0.3540` vs `0.3558` (`-0.0018`)
  - `mAP50-95`: `0.1648` vs `0.1658` (`-0.0010`)
- Speed gain vs `.pt` is significant:
  - `fps`: `107.16` vs `79.97` (about `1.34x` faster)
  - `latency`: `9.332 ms` vs `12.505 ms`
- `.onnx (fp16)` does not outperform `.onnx` FP32 in this setup (`78.05 fps` vs `107.16 fps`), so it is not the optimal choice here.

Practical trade-off:

- Use `.onnx` FP32 as the primary edge deployment artifact for real-time inference.
- Keep `.pt` as the quality baseline to monitor accuracy regression in future model iterations.

---

## Підсумки

**Вибір для production edge-inference: `.onnx` (FP32).**

Чому:

- У поточному бенчмарку `.onnx` має найкращу швидкодію (мінімальна latency та максимальний FPS).
- Просідання якості відносно `.pt` невелике:
  - `mAP50`: `0.3540` vs `0.3558` (`-0.0018`)
  - `mAP50-95`: `0.1648` vs `0.1658` (`-0.0010`)
- Приріст швидкості відносно `.pt` суттєвий:
  - `fps`: `107.16` vs `79.97` (приблизно `1.34x` швидше)
  - `latency`: `9.332 ms` vs `12.505 ms`
- `.onnx (fp16)` у цьому сетапі не швидший за `.onnx` FP32 (`78.05 fps` vs `107.16 fps`), тому не є оптимальним вибором тут.

Практичний компроміс:

- Використовуємо `.onnx` FP32 як основний edge-артефакт для real-time інференсу.
- Залишаємо `.pt` як quality baseline для контролю можливої деградації якості в наступних ітераціях моделі.